# Notebook 2: Molecular filtering
Our objective is to filter our dataset down to only high-quality, "drug-like" molecules.

We achieve this through two specialized filtering mechanisms:
1. Bioavailability Profiling (Lipinski’s Rule of Five): We calculate four core physicochemical descriptors (Molecular Weight, LogP, Hydrogen Bond Donors, and Acceptors).
2. Structural Sanity Checks (PAINS & Toxicophores): We utilize RDKit and pre-defined SMARTS patterns to screen the dataset for Pan Assay Interference Compounds (PAINS)—notorious chemical tricksters that cause false positives in lab tests. Additionally, we filter out known "unwanted substructures" (toxicophores) to remove highly reactive, mutagenic, or unstable compounds from our training space.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, PandasTools
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

In [ ]:
HERE = Path.cwd()
DATA = HERE / "data"

### 1. Calculating Lipinski's Rule of Five Descriptors

To evaluate the drug-likeness of our Androgen Receptor inhibitors, we will calculate four key physicochemical properties defined by Lipinski's Rule of Five (Ro5):
1. Molecular Weight (MW): <= 500 Daltons
2. Number of Hydrogen Bond Acceptors (HBA): <= 10
3. Number of Hydrogen Bond Donors (HBD): <= 5
4. Octanol-Water Partition Coefficient (LogP): <= 5

In traditional drug discovery, a molecule is generally considered orally bioavailable (suitable for a pill) if it violates no more than one of these four rules.

In the following steps, we will:
* Define a custom RDKit function to calculate these specific metrics directly from our SMILES strings.
* Evaluate the Ro5 conditions and create a boolean flag (`ro5_fulfilled`) that returns `True` if the molecule has 1 or fewer violations.

In [ ]:
# Import the AR dataset from Notebook 1
molecules = pd.read_csv(DATA / "AR_compounds.csv", index_col=0)
molecules.head()

In [ ]:
# Define a function to calculate the Lipinski's Ro5 descriptors
def calculate_ro5_properties(smiles):

    # RDKit molecule from SMILES
    molecule = Chem.MolFromSmiles(smiles)

    # Calculate Ro5-relevant chemical properties
    molecular_weight = Descriptors.ExactMolWt(molecule)
    n_hba = Descriptors.NumHAcceptors(molecule)
    n_hbd = Descriptors.NumHDonors(molecule)
    logp = Descriptors.MolLogP(molecule)

    # Check if Ro5 conditions fulfilled
    conditions = [molecular_weight <= 500, n_hba <= 10, n_hbd <= 5, logp <= 5]
    ro5_fulfilled = sum(conditions) >= 3

    # Return True if no more than one out of four conditions is violated
    return pd.Series(
        [molecular_weight, n_hba, n_hbd, logp, ro5_fulfilled],
        index=["molecular_weight", "n_hba", "n_hbd", "logp", "ro5_fulfilled"],
    )

In [ ]:
# Apply the function to the AR dataset
ro5_properties = molecules["smiles"].apply(calculate_ro5_properties)
ro5_properties.head()

In [ ]:
# Merge the Lipinski's Ro5 results into the AR dataset
molecules = pd.concat([molecules, ro5_properties], axis=1)
molecules.head()

In [ ]:
print("Dataframe shape:", molecules.shape)

### 2. Filter PAINs and Unwanted Structures
With our physicochemical properties calculated, the final step in our data curation pipeline is to remove problematic compounds that could mislead our machine learning model.

In this section, we apply two strict structural filters:
* PAINS Filter: We utilize RDKits built-in FilterCatalog to screen for Pan Assay Interference Compounds. These are molecules known to react non-specifically in biological assays, yielding false positive results.
* Unwanted Substructures Filter: We load a predefined list of reactive, toxic, or undesirable chemical groups and remove any molecules containing these substructures.

After stripping away these problematic compounds, we are left with a highly curated, drug-like dataset. We then save this finalized dataset to a new CSV file, which will serve as the clean input for our machine learning training.

In [ ]:
# Initialize a blank parameters object to configure our structural
params = FilterCatalogParams()

# Add the official PAINS (Pan Assay Interference Compounds) catalog to our parameters.
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)

# Build the active FilterCatalog engine using our PAINS parameters.
# We will use this 'catalog' object to scan our molecules and flag any rule-breakers.
catalog = FilterCatalog(params)

In [ ]:
# Initialize empty lists to separate the problematic molecules from the safe ones
matches_PAIN = []
clean_PAIN = []

# Iterate through the dataset using tqdm to display a progress bar
for index, row in tqdm(molecules.iterrows(), total=molecules.shape[0]):

    # Convert the 1D SMILES string into a 2D RDKit molecule object for structural scanning
    molecule = Chem.MolFromSmiles(row.smiles)

    # Screen the molecule against the PAINS catalog
    entry = catalog.GetFirstMatch(molecule)  # Get the first matching PAINS
    if entry is not None:
        # If a PAINS substructure is found, record its details
        matches_PAIN.append(
            {
                "chembl_id": row.molecule_chembl_id,
                "rdkit_molecule": molecule,
                "pains": entry.GetDescription().capitalize(),
            }
        )
    else:
        # If no PAINS are found, save the index of this molecule
        clean_PAIN.append(index)

# Convert the flagged PAINS into their own dataframe
matches_PAIN = pd.DataFrame(matches_PAIN)

# Overwrite our working dataframe to strictly include only the safe, PAINS-free molecules
molecules = molecules.loc[clean_PAIN]

In [ ]:
# Check output
print(f"Number of compounds with PAINS: {len(matches_PAIN)}")
print(f"Number of compounds without PAINS: {len(molecules)}")

In [ ]:
# Load the curated list of unwanted chemical substructures
substructures = pd.read_csv(DATA / "unwanted_substructures.csv", sep=r"\s+")

# Convert the 1D SMARTS strings (chemical query patterns) into 2D RDKit objects
substructures["rdkit_molecule"] = substructures.smarts.apply(Chem.MolFromSmarts)

# Check number of unwanted substructures in the list
print("Number of unwanted substructures in collection:", len(substructures))

In [ ]:
# Initialize lists to separate the clean molecules from those containing unwanted groups
matches_sub = []
clean_sub = []

# Iterate through the dataset using tqdm to display a progress bar
for index, row in tqdm(molecules.iterrows(), total=molecules.shape[0]):

    # Convert the 1D SMILES string into a 2D RDKit molecule object
    molecule = Chem.MolFromSmiles(row.smiles)
    match = False

    # Inner loop: Check the current molecule against EVERY unwanted substructure in our list
    for _, substructure in substructures.iterrows():

        # HasSubstructMatch performs a graph-matching algorithm to see if the unwanted fragment exists anywhere inside our target molecule
        if molecule.HasSubstructMatch(substructure.rdkit_molecule):

            # If a match is found, record the specific toxic/reactive group it hit
            matches_sub.append(
                {
                    "chembl_id": row.molecule_chembl_id,
                    "rdkit_molecule": molecule,
                    "substructure": substructure.rdkit_molecule,
                    "substructure_name": substructure["name"],
                }
            )
            match = True

    # If the molecule survived the entire inner loop without a single match, it is safe
    if not match:
        clean_sub.append(index)

# Save the problematic molecules to a separate dataframe
matches_sub = pd.DataFrame(matches_sub)

# Finalize the dataset by overwriting it with only the structurally safe compounds
molecules = molecules.loc[clean_sub]

In [ ]:
# Check output
print(f"Number of found unwanted substructure: {len(matches_sub)}")
print(f"Number of compounds without unwanted substructure: {len(molecules)}")

In [ ]:
# Reset index of the data frame
molecules.reset_index(drop=True, inplace=True)

In [ ]:
# Look at head
molecules.head()

In [ ]:
# Save the final filtered dataset to csv file
molecules.to_csv(DATA / "AR_filtered.csv")
print("Dataframe shape:", molecules.shape)